Intermediate IP flows by class
==============================

Using WIPO IP indicators: PA5, TM4a/b, ID4a/b

| **IP Type** | **Code** | **Columns** | **Description** |
|---|---|---|---|
| **Patents** | PA4a  | Office, Office (Code), Origin, Field of technology, year, count | Patent publications by technology |
| **Patents** | PA5  | Office, Office (Code), Origin, Field of technology, year, count | Patent grants by technology |
| **Trademarks** | TM3a | Office, Office (Code), Origin, Nice classification, year, count | Direct applications by class|
| **Trademarks** | TM3b | Office, Office (Code), Origin, Nice classification, year, count | Applications via the Madrid system by class|
| **Trademarks** | TM4a | Office, Office (Code), Origin, Nice classification, year, count | Registrations for direct applications by class |
| **Trademarks** | TM4b | Office, Office (Code), Origin, Nice classification, year, count | Registrations for applications via the Madrid system by class|
| **Industrial designs** | ID3a | Office, Office (Code), Origin, Locarno classification, year, count | Direct applications by class |
| **Industrial designs** | ID3b | Office, Office (Code), Origin, Locarno classification, year, count | Applications via the Hague system by class|
| **Industrial designs** | ID4a | Office, Office (Code), Origin, Locarno classification, year, count | Registrations for direct applications by class |
| **Industrial designs** | ID4b | Office, Office (Code), Origin, Locarno classification, year, count | Registrations for applications via the Hague system by class |


In [1]:
from pathlib import Path
import pandas as pd
import sys
import re

In [2]:
current_dir = Path.cwd()

# Input: Staging Data
staging_path = (current_dir.parent.parent / "data" / "staging" / "wipo" / "ip_indicators").resolve()

# Input: Class Labels (Nice/Locarno definitions)
class_labels_path = (current_dir.parent.parent / "data" / "staging" / "wipo" / "ip_classes").resolve()

# Output: Intermediate Data (Target Folder)
intermediate_path = (current_dir.parent.parent / "data" / "intermediate").resolve()
intermediate_path.mkdir(parents=True, exist_ok=True)

output_file = intermediate_path / "int_ip_flows_by_class.parquet"

# Label Files
NICE_XLSX = class_labels_path / "nice" / "Nice.xlsx"
LOCARNO_XLSX = class_labels_path / "locarno" / "locarno.xlsx"

print(f"Reading Staging from: {staging_path}")
print(f"Reading Labels from:  {class_labels_path}")
print(f"Exporting to:         {output_file}")

Reading Staging from: C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\staging\wipo\ip_indicators
Reading Labels from:  C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\staging\wipo\ip_classes
Exporting to:         C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\intermediate\int_ip_flows_by_class.parquet


In [3]:
target_files = [
    "stg_wipo__PA4a.parquet", "stg_wipo__PA5.parquet",
    "stg_wipo__TM3a.parquet", "stg_wipo__TM3b.parquet", 
    "stg_wipo__TM4a.parquet", "stg_wipo__TM4b.parquet",
    "stg_wipo__ID3a.parquet", "stg_wipo__ID3b.parquet", 
    "stg_wipo__ID4a.parquet", "stg_wipo__ID4b.parquet"
]

In [4]:
# Helper Functions
def clean_class_code(code):
    """Normalize 'Class 07' -> '07', keep 'Other'/'Unknown'."""
    s = str(code).strip()
    s_lower = s.lower()
    if "other" in s_lower: return "Other"
    if "unknown" in s_lower: return "Unknown"
    
    # Extract number if present (e.g. "Class 07" -> "07")
    match = re.search(r"(\d+)", s)
    if match:
        return f"{int(match.group(1)):02d}"
    return s

def load_label_map(path):
    """Returns dict: {'01': '01 - Chemical products', ...}"""
    if not path.exists():
        print(f"[WARN] Label file not found: {path}")
        return {}
    
    try:
        df = pd.read_excel(path)
        df.columns = [c.strip().lower() for c in df.columns] # normalize headers
        
        # Expecting columns 'class' and 'industry' (or similar)
        if 'class' in df.columns and 'industry' in df.columns:
            df['clean_code'] = df['class'].apply(clean_class_code)
            df['label'] = df['clean_code'] + " - " + df['industry'].astype(str).str.strip()
            return dict(zip(df['clean_code'], df['label']))
        else:
            print(f"[WARN] Columns 'class' and 'industry' not found in {path.name}")
            return {}
            
    except Exception as e:
        print(f"[WARN] Failed to load labels {path}: {e}")
        return {}

In [5]:
nice_map = load_label_map(NICE_XLSX)
locarno_map = load_label_map(LOCARNO_XLSX)

In [6]:
print(nice_map)

{'01': '01 - Chemicals', '02': '02 - Paints & Coatings', '03': '03 - Cosmetics & Cleaning Preps', '04': '04 - Oils, Lubricants & Fuels', '05': '05 - Pharmaceuticals & Medicated Goods', '06': '06 - Metals & Metal Goods', '07': '07 - Machinery & Machine Tools', '08': '08 - Hand Tools', '09': '09 - Scientific & Electronic Apparatus', '10': '10 - Medical Devices', '11': '11 - Lighting, Heating & HVAC', '12': '12 - Vehicles & Locomotion', '13': '13 - Firearms & Explosives', '14': '14 - Jewelry & Precious Metals', '15': '15 - Musical Instruments', '16': '16 - Paper, Printing & Stationery', '17': '17 - Rubber, Plastics & Insulation (semi-finished)', '18': '18 - Leather Goods & Luggage', '19': '19 - Non-Metal Building Materials', '20': '20 - Furniture & Home Goods', '21': '21 - Household Utensils & Glassware', '22': '22 - Ropes, Nets & Raw Fibers', '23': '23 - Yarns & Threads', '24': '24 - Fabrics & Home Textiles', '25': '25 - Clothing, Footwear & Headwear', '26': '26 - Haberdashery (Lace, Rib

In [7]:
print(locarno_map)

{'01': '01 - Foodstuffs', '02': '02 - Clothing & Haberdashery', '03': '03 - Luggage & Travel Goods', '04': '04 - Brushes & Brushware', '05': '05 - Textile Fabrics & Yardage', '06': '06 - Furniture & Furnishings', '07': '07 - Household Goods', '08': '08 - Tools & Hardware', '09': '09 - Packaging & Containers', '10': '10 - Timepieces', '11': '11 - Jewelry & Adornments', '12': '12 - Vehicles & Hoisting', '13': '13 - Electrical Equipment', '14': '14 - Audio/Video & Communications', '15': '15 - Industrial Machines', '16': '16 - Photo & Optical Apparatus', '17': '17 - Musical Instruments', '18': '18 - Printing & Office Machines', '19': '19 - Stationery & Office Supplies', '20': '20 - Retail Displays & Signs', '21': '21 - Toys, Games & Sporting Goods', '22': '22 - Arms, Hunting & Fishing', '23': '23 - Fluid Handling Equipment', '24': '24 - Medical & Laboratory Equipment', '25': '25 - Building Components', '26': '26 - Lighting Appliances', '27': '27 - Tobacco & Smoking Articles', '28': '28 - P

In [8]:
#  Main Loop 
dfs_to_combine = []

for filename in target_files:
    file_path = staging_path / filename
    if not file_path.exists():
        print(f"[MISSING] {filename}")
        continue

    try:
        df = pd.read_parquet(file_path)
        
        # DETERMINE TYPE BY FILENAME 
        if "PA" in filename:
            class_type = 'Tech field'
            label_map = {} 
            # FIX: Do not clean patents, they already have the format "ID - Description"
            skip_cleaning = True   
        elif "TM" in filename:
            class_type = 'Nice'
            label_map = nice_map
            skip_cleaning = False
        elif "ID" in filename:
            class_type = 'Locarno'
            label_map = locarno_map
            skip_cleaning = False
        else:
            class_type = 'Unknown'
            label_map = {}
            skip_cleaning = False

        if 'class_code' not in df.columns:
             print(f"[SKIP] {filename} - 'class_code' column missing")
             continue

        #  Normalize Class Code (ONLY if not skipping)
        if not skip_cleaning:
            df['class_code'] = df['class_code'].apply(clean_class_code)
        
        # Apply Labels (if applicable)
        if label_map:
            # .fillna() is crucial here!
            df['class_code'] = df['class_code'].map(label_map).fillna(df['class_code'])

        # Add Metadata
        df['indicator'] = filename.replace("stg_wipo__", "").replace(".parquet", "")
        df['class_type'] = class_type
        
        # Melt Years (Wide -> Long)
        # Identify ID variables (everything that isn't a year)
        # Note: Ensure these match your staging column names exactly (Office vs office)
        id_vars = ['Office', 'Origin', 'class_code', 'indicator', 'class_type']
        
        # Dynamic value vars: everything else (Years)
        value_vars = [c for c in df.columns if c not in id_vars]
        
        df_melted = df.melt(
            id_vars=id_vars,
            value_vars=value_vars,
            var_name='year',
            value_name='count'
        )

        #  Clean Numeric Data
        df_melted['year'] = pd.to_numeric(df_melted['year'], errors='coerce') 
        df_melted['count'] = pd.to_numeric(df_melted['count'], errors='coerce')
        
        # Drop invalid rows (e.g. Year='Unnamed: 49' or Count='NaN')
        df_melted = df_melted.dropna(subset=['year', 'count'])
        
        dfs_to_combine.append(df_melted)
        print(f"Processed: {filename} ({class_type})")

    except Exception as e:
        print(f"[ERROR] {filename}: {e}")

Processed: stg_wipo__PA4a.parquet (Tech field)
Processed: stg_wipo__PA5.parquet (Tech field)
Processed: stg_wipo__TM3a.parquet (Nice)
Processed: stg_wipo__TM3b.parquet (Nice)
Processed: stg_wipo__TM4a.parquet (Nice)
Processed: stg_wipo__TM4b.parquet (Nice)
Processed: stg_wipo__ID3a.parquet (Locarno)
Processed: stg_wipo__ID3b.parquet (Locarno)
Processed: stg_wipo__ID4a.parquet (Locarno)
Processed: stg_wipo__ID4b.parquet (Locarno)


In [9]:
if dfs_to_combine:
    final_df = pd.concat(dfs_to_combine, ignore_index=True)
    
    # Standardize column names
    final_df.columns = final_df.columns.str.lower()
    
    # Reorder columns for readability
    final_df = final_df[['year', 'origin', 'office', 'indicator', 'class_type', 'class_code', 'count']]
    
    final_df.to_parquet(output_file, index=False)
    print(f"\nCompleted. Saved {len(final_df):,} rows to: {output_file}")
    
    # Validation Print
    print(final_df.sample(5).to_string(index=False))
else:
    print("No data processed.")


Completed. Saved 8,786,629 rows to: C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\intermediate\int_ip_flows_by_class.parquet
 year         origin            office indicator class_type                        class_code  count
 2020        Türkiye       Switzerland      TM3b       Nice    14 - Jewelry & Precious Metals    4.0
 2007 United Kingdom           Ireland      TM3b       Nice 19 - Non-Metal Building Materials    5.0
 2023       Slovenia   North Macedonia      TM3b       Nice      24 - Fabrics & Home Textiles    4.0
 2017        Ukraine Republic of Korea      TM4b       Nice         06 - Metals & Metal Goods    1.0
 2010   South Africa         Singapore      TM3a       Nice 16 - Paper, Printing & Stationery    2.0
